**Goal**
- We need to evaluate the error of the predictions they are making using the "Last-Year-Same-Week" heuristic.
    - For business needs, **Mean Absolute Percentage Error (MAPE)** and **Mean Absolute Error (MAE)** are the most interpretable for stakeholders.
    - However, we want to severely penalize forecasting errors made on a high-demand holiday compared to a normal sales day.
    - For that, we will use the `IsHoliday` feature to provide a multiplying weight to our errors.
    - Ultimately, our error metrics will be:
        - **Weighted MAE (WMAE):** The average absolute dollar error. 
          $\frac{\sum (|Actual - Predicted| \times Weight)}{\sum Weight}$
        - **Weighted MAPE (WMAPE):** The percentage error relative to actual sales. 
          $\frac{\sum (|Actual - Predicted| \times Weight)}{\sum (|Actual| \times Weight)}$


### SUMMARY: Building the Weighted Formulas
To get our weighted errors, we first need to decide how much more "expensive" a mistake is during a holiday week. In retail, the standard rule of thumb is that **holiday errors are penalized 5 times more than normal weeks** because stock-outs during major holiday rushes destroy customer trust and revenue.

**How the math translates to code:**
1. We assign a `Weight` of `5` for holidays, and `1` for normal weeks.
2. **WMAE** tells us the average dollar mistake we make per week, per department, heavily skewing toward holiday failures.
3. **WMAPE** divides our total weighted error by our total weighted actual sales. Doing it this way completely avoids the classic "divide by zero" computer error if a department happens to have $0 in sales for a week, giving us a clean, stable percentage to report to executives.

Imagine just two weeks of data:

- Week 1 (Normal): Actual = $100. Error = $50. (Weight = 1)
- Week 2 (Holiday): Actual = $1,000. Error = $10. (Weight = 5)
- Without Weights: 
  - Total Error = 50 + 10 = $60 Total Actual = 100 + 1,000 = $1,100 
  - Total Error % = (60 / 1100) * 100= 5.4% Error

- With Weights: 
  - Numerator = (1 * 50) + (5 * 10) = 50 + 50 = $100 
  - Denominator = (1 * 100) + (5 * 1,000) = 100 + 5,000 = $5,100 
  - Weighted Error = (100 / 5100) * 100 = 1.9% Error

In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from pathlib import Path
import sys

# Get the absolute path of the parent folder
parent_dir = str(Path().resolve().parent)

if parent_dir not in sys.path:
    sys.path.append(parent_dir)


In [3]:
import pandas as pd 
import numpy as np
import config

In [4]:
raw_store_sales = pd.read_csv(config.RAW_SALES_PATH)
raw_store_features = pd.read_csv(config.RAW_FEATURES_PATH)
raw_stores = pd.read_csv(config.RAW_STORES_PATH)

display(raw_store_sales.head())
display(raw_store_features.head())
display(raw_stores.head())

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,05/02/2010,24924.50,False
1,1,1,12/02/2010,46039.49,True
2,1,1,19/02/2010,41595.55,False
3,1,1,26/02/2010,19403.54,False
4,1,1,05/03/2010,21827.90,False


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,05/02/2010,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,12/02/2010,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,19/02/2010,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,26/02/2010,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,05/03/2010,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [5]:
# the date is in day/month/year format i.e d/m/Y
raw_store_sales['Date'] = pd.to_datetime(raw_store_sales['Date'], format='%d/%m/%Y')
raw_store_sales = raw_store_sales.sort_values(by=['Date', 'Store'], ascending=True).reset_index(drop=True)
raw_store_sales

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,2,2010-02-05,50605.27,False
2,1,3,2010-02-05,13740.12,False
3,1,4,2010-02-05,39954.04,False
4,1,5,2010-02-05,32229.38,False
...,...,...,...,...,...
421565,45,93,2012-10-26,2487.80,False
421566,45,94,2012-10-26,5203.31,False
421567,45,95,2012-10-26,56017.47,False
421568,45,97,2012-10-26,6817.48,False


In [6]:
display(raw_store_sales['Date'].describe())


count                           421570
mean     2011-06-18 08:30:31.963375360
min                2010-02-05 00:00:00
25%                2010-10-08 00:00:00
50%                2011-06-17 00:00:00
75%                2012-02-24 00:00:00
max                2012-10-26 00:00:00
Name: Date, dtype: object

- Move back in time to show the sales same week week previous year
- This didnt work as expected because the dates are not matching, there maybe some skipped week and the starting date for 2010 is 5 whereas for 2011 its 08.
- So, we will need to use self join

In [7]:
# LAG (Past to Current): To predict the current week, look 52 weeks back

lysw_df = raw_store_sales.copy()
lysw_df['LYSW_Sales'] = lysw_df.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(52)

lysw_df = lysw_df[['Date','Store','Dept', 'Weekly_Sales', 'LYSW_Sales']]
lysw_df[lysw_df['Date']>='2011-01-01']

,Date,Store,Dept,Weekly_Sales,LYSW_Sales
140679,2011-01-07,1,1,15984.24,NaN
140680,2011-01-07,1,2,43202.29,NaN
140681,2011-01-07,1,3,15808.15,NaN
140682,2011-01-07,1,4,37947.80,NaN
140683,2011-01-07,1,5,22699.69,NaN
...,...,...,...,...,...
421565,2012-10-26,45,93,2487.80,1600.80
421566,2012-10-26,45,94,5203.31,5052.12
421567,2012-10-26,45,95,56017.47,52619.53
421568,2012-10-26,45,97,6817.48,5645.89


**Self join approach**

In [8]:
# Current sales data
past_data = raw_store_sales[['Store', 'Dept', 'Date', 'Weekly_Sales']].copy()

# Add 52 weeks to the current dates , everything else remains the same 
past_data['Date'] = past_data['Date'] + pd.Timedelta(weeks=52)

past_data = past_data.rename(columns={'Weekly_Sales':'Baseline_Prediction'})
display(past_data.head(5))

,Store,Dept,Date,Baseline_Prediction
0,1,1,2011-02-04,24924.50
1,1,2,2011-02-04,50605.27
2,1,3,2011-02-04,13740.12
3,1,4,2011-02-04,39954.04
4,1,5,2011-02-04,32229.38


* Now we take our current actual sales and merge our past baseline predictions onto them. Any matching dates will yield us the actual sales for that week alongside what the heuristic predicted.

In [9]:
baseline_predictions = raw_store_sales.merge(past_data, on=['Store','Dept','Date'], how='left')

# drop the 2010 dates , because we dont have 2009 data for them to generate predictions
baseline_predictions = baseline_predictions.dropna(subset=['Baseline_Prediction'])

baseline_predictions.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Baseline_Prediction
152374,1,1,2011-02-04,21665.76,False,24924.50
152375,1,2,2011-02-04,46829.12,False,50605.27
152376,1,3,2011-02-04,11012.52,False,13740.12
152377,1,4,2011-02-04,35870.49,False,39954.04
152378,1,5,2011-02-04,31280.62,False,32229.38


- Now we just need to find out the error between `Weekly_Sales` and `Baseline_Prediction` weighted using `IsHoliday`

In [10]:
# RAW Absolute error
baseline_predictions['ABS_Error'] = abs(baseline_predictions['Weekly_Sales'] - baseline_predictions['Baseline_Prediction'])


In [11]:
baseline_predictions['Weight'] = np.where(baseline_predictions['IsHoliday'], 5, 1)
baseline_predictions.sample(10)
# True holiday samples are rare

,Store,Dept,Date,Weekly_Sales,IsHoliday,Baseline_Prediction,ABS_Error,Weight
271590,21,95,2011-11-11,36990.24,False,33105.96,3884.28,1
172338,34,85,2011-03-18,1300.16,False,1191.82,108.34,1
340732,29,52,2012-04-20,1044.58,False,1232.24,187.66,1
319005,16,38,2012-03-02,57489.30,False,45954.63,11534.67,1
289685,23,23,2011-12-23,116157.84,False,121981.46,5823.62,1
165476,20,50,2011-03-04,4638.36,False,4814.87,176.51,1
225232,34,20,2011-07-22,2556.99,False,2857.47,300.48,1
363114,11,52,2012-06-15,2696.35,False,1483.56,1212.79,1
182492,10,96,2011-04-15,10336.37,False,9050.97,1285.40,1
163878,41,51,2011-02-25,1.50,False,48.24,46.74,1


In [12]:
wmae = ((baseline_predictions['Weight'] *
        baseline_predictions['ABS_Error']).sum() /
        baseline_predictions['Weight'].sum())
print(wmae)

1969.9087942713606


In [13]:
wmape = (((baseline_predictions['Weight'] * 
        baseline_predictions['ABS_Error']).sum()) /
        (baseline_predictions['Weight'] * 
        baseline_predictions['Weekly_Sales']).sum())
print(wmape)

0.11846852635329808


In [14]:
print(f"WMAE: {wmae:.2f}")
print(f"WMAPE: {(wmape * 100):.2f}%")

WMAE: 1969.91
WMAPE: 11.85%
